## Generating a Dummy Dataset 

In [8]:
# !pip install khmercut pycrfsuite pytorch-lightning mlflow    # Install required libraries

# Dummy Khmer sentences (in Khmer script)
khmer_sentences = [
    "សួស្តី ពិភពលោក",                         # "Hello world"
    "ខ្ញុំស្រលាញ់ កម្ពុជា",                     # "I love Cambodia"
    "ក្រុង​ភ្នំពេញ គឺ​ជា រណ្តៅ​សេដ្ឋកិច្ច",        # "Phnom Penh is an economic hub"
    "កូន​ខ្មែរ សិក្សា ភាសា អង់គ្លេស នៅ សាលា",    # "Khmer kids study English at school"
    "អ្នកគ្រូ បង្រៀន ភាសាខ្មែរ នៅ​ក្នុង ថ្នាក់",   # "The teacher teaches Khmer in class"
    "ប្រទេស កម្ពុជា មាន បេតិកភណ្ឌ សម្បូរ",       # "Cambodia has abundant heritage"
    "ប្រាសាទអង្គរ វត្ត មាន ល្បីឈ្មោះ នៅលើ លោក", # "Angkor Wat temple is famous in the world"
    "បងប្រុស របស់​ខ្ញុំ ស្ថិតនៅទីក្រុង​ប៉ារីស",    # "My brother is in Paris"
    "ការពិត ជា ក្តៅ ច្រើន ក្នុង រដូវ​ក្តៅ",       # "It is very hot in summer"
    "ខ្ញុំ ចូលចិត្ត ញ៉ាំ កាហ្វេ ពេល ព្រឹក"         # "I like to drink coffee in the morning"
]

# Dummy English sentences
english_sentences = [
    "Hello world",
    "I am writing code",
    "This is a short English sentence",
    "PyTorch Lightning makes training easier",
    "We are creating a bilingual model",
    "Artificial intelligence is evolving rapidly",
    "Cambodia is a beautiful country",
    "Natural language processing is interesting",
    "He went to the market",
    "She enjoys reading books"
]

print(f"Number of Khmer sentences: {len(khmer_sentences)}")
print(f"Number of English sentences: {len(english_sentences)}")
print("Example Khmer sentence:", khmer_sentences[0])
print("Example English sentence:", english_sentences[0])


Number of Khmer sentences: 10
Number of English sentences: 10
Example Khmer sentence: សួស្តី ពិភពលោក
Example English sentence: Hello world


## Building a Custom Tokenizer and Vocabulary

In [9]:
from khmercut import tokenize as khmer_tokenize

class CustomTokenizer:
    def __init__(self, vocab=None):
        self.vocab = vocab or {}
        # Build inverse vocab for decoding
        self.id_to_token = {idx: tok for tok, idx in self.vocab.items()} if vocab else {}
    
    def build_vocab(self, texts, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]):
        """Build vocabulary from a list of texts. Returns the vocab dict."""
        # Use a set to collect tokens
        token_set = set()
        for text in texts:
            tokens = self.tokenize(text)
            token_set.update(tokens)
        # Start vocab with special tokens
        vocab_list = list(special_tokens)
        vocab_list += sorted(token_set - set(special_tokens))
        self.vocab = {tok: i for i, tok in enumerate(vocab_list)}
        self.id_to_token = {i: tok for tok, i in self.vocab.items()}
        return self.vocab

    def tokenize(self, text):
        """Tokenize a given text into a list of tokens (Khmer or English)."""
        try:
            tokens = khmer_tokenize(text)  # Khmer segmentation (works for mixed text too)
        except Exception as e:
            # Fallback to simple whitespace split if khmercut is unavailable
            tokens = text.split()
        # Filter out any empty tokens or pure space tokens that may result from segmentation
        tokens = [tok for tok in tokens if tok.strip() != ""]
        return tokens

    def tokens_to_ids(self, tokens):
        """Convert list of tokens to list of vocabulary IDs."""
        return [self.vocab.get(tok, self.vocab.get("[UNK]")) for tok in tokens]

    def ids_to_tokens(self, ids):
        """Convert list of IDs to list of tokens."""
        return [self.id_to_token.get(i, "[UNK]") for i in ids]

    def encode(self, text_a, text_b=None, add_special_tokens=True):
        """
        Encode one or two sentences into token IDs, adding [CLS], [SEP] tokens.
        Returns a list of token IDs and a list of token_type_ids (segment ids).
        """
        tokens_a = self.tokenize(text_a)
        tokens_b = self.tokenize(text_b) if text_b is not None else None
        if add_special_tokens:
            tokens = ["[CLS]"] + tokens_a + ["[SEP]"]
            token_type_ids = [0] * len(tokens)
            if tokens_b is not None:
                tokens += tokens_b + ["[SEP]"]
                token_type_ids += [1] * (len(tokens_b) + 1)  # segment 1 for B and its [SEP]
        else:
            tokens = tokens_a + (tokens_b if tokens_b else [])
            token_type_ids = [0] * len(tokens_a) + ([1] * len(tokens_b) if tokens_b else [])
        input_ids = self.tokens_to_ids(tokens)
        return input_ids, token_type_ids

    def decode(self, ids, skip_special_tokens=True):
        """Convert IDs back to string (for inference)."""
        tokens = self.ids_to_tokens(ids)
        if skip_special_tokens:
            tokens = [t for t in tokens if t not in ["[CLS]", "[SEP]", "[PAD]", "[MASK]"]]
        # Join tokens with space (Khmer tokens will be joined by spaces which visually is fine for output)
        return " ".join(tokens)

# Initialize tokenizer and build vocabulary from all sentences
tokenizer = CustomTokenizer()
all_texts = khmer_sentences + english_sentences
vocab = tokenizer.build_vocab(all_texts)
print(f"Vocab size: {len(vocab)}")
# Show some sample mappings
print("Token for ID 0:", tokenizer.id_to_token[0])
print("ID for token 'កម្ពុជា' (Cambodia in Khmer):", vocab.get("កម្ពុជា"))
print("ID for token 'Hello':", vocab.get("Hello"))
# Test encoding
sample_text = "ខ្ញុំស្រលាញ់ កម្ពុជា"  # "I love Cambodia"
ids, type_ids = tokenizer.encode(sample_text)
print("Sample text:", sample_text)
print("Tokenized:", tokenizer.tokenize(sample_text))
print("Encoded IDs:", ids)


Vocab size: 96
Token for ID 0: [PAD]
ID for token 'កម្ពុជា' (Cambodia in Khmer): 47
ID for token 'Hello': 9
Sample text: ខ្ញុំស្រលាញ់ កម្ពុជា
Tokenized: ['ខ្ញុំ', 'ស្រលាញ់', 'កម្ពុជា']
Encoded IDs: [2, 54, 91, 47, 3]


## Creating a Pretraining DataLoader (MLM + NSP)

In [10]:
import math, random
import torch
from torch.utils.data import Dataset, DataLoader

class PretrainingDataset(Dataset):
    def __init__(self, sentences_a, sentences_b, tokenizer, max_seq_length=64):
        """
        Prepare sentence pairs for NSP. sentences_a and sentences_b are lists of sentences.
        We'll treat each list as a "document" and create positive (next) pairs from within the same list, 
        and negative (random) pairs from different lists.
        """
        self.tokenizer = tokenizer
        self.max_len = max_seq_length
        self.samples = []  # will hold tuples of (text_a, text_b, is_next_label)
        # Create positive pairs (IsNext = 1)
        for i in range(len(sentences_a) - 1):
            self.samples.append((sentences_a[i], sentences_a[i+1], 1))
        for j in range(len(sentences_b) - 1):
            self.samples.append((sentences_b[j], sentences_b[j+1], 1))
        # Create negative pairs (IsNext = 0) of equal count
        num_positive = len(self.samples)
        random.shuffle(sentences_a)
        random.shuffle(sentences_b)
        all_sentences = sentences_a + sentences_b
        for k in range(num_positive):
            # For each positive pair, pick a random other sentence for negative pair
            text_a, text_b_pos, label = self.samples[k]
            # Ensure the random sentence is not actually the true next sentence
            rand_b = random.choice(all_sentences)
            # If by chance it's the same as true next, choose another
            if k < len(sentences_a)-1 and rand_b == sentences_a[k+1]:
                rand_b = random.choice(all_sentences)
            if k >= len(sentences_a) and k-len(sentences_a) < len(sentences_b)-1 and rand_b == sentences_b[k-len(sentences_a)+1]:
                rand_b = random.choice(all_sentences)
            self.samples.append((text_a, rand_b, 0))
        random.shuffle(self.samples)  # shuffle the order of samples

        # Special token IDs for convenience
        self.cls_id = tokenizer.vocab["[CLS]"]
        self.sep_id  = tokenizer.vocab["[SEP]"]
        self.mask_id = tokenizer.vocab["[MASK]"]
        self.pad_id  = tokenizer.vocab["[PAD]"]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text_a, text_b, is_next = self.samples[idx]
        # Encode the text pair into token ids and segment ids
        input_ids, token_type_ids = self.tokenizer.encode(text_a, text_b, add_special_tokens=True)
        # Truncate if too long
        if len(input_ids) > self.max_len:
            input_ids = input_ids[:self.max_len]
            token_type_ids = token_type_ids[:self.max_len]
            # Make sure last token is [SEP] if truncated
            input_ids[-1] = self.sep_id
            token_type_ids[-1] = 1 if text_b is not None else 0
        # Create attention mask (1 for real tokens, 0 for pads)
        attention_mask = [1] * len(input_ids)
        # MLM: determine positions to mask
        num_to_mask = max(1, math.floor(0.15 * len(input_ids)))
        # Do not mask special tokens ([CLS], [SEP], etc.)
        candidate_positions = [i for i, token_id in enumerate(input_ids) 
                                if token_id not in (self.cls_id, self.sep_id, self.pad_id)]
        mask_positions = random.sample(candidate_positions, min(num_to_mask, len(candidate_positions)))
        # Prepare MLM labels, initialize all to -100 (ignore)
        mlm_labels = [-100] * len(input_ids)
        for pos in mask_positions:
            original_token = input_ids[pos]
            mlm_labels[pos] = original_token  # store the target
            # Replace input token with [MASK], random, or keep original
            rand = random.random()
            if rand < 0.8:
                # 80% replace with [MASK]
                input_ids[pos] = self.mask_id
            elif rand < 0.9:
                # 10% replace with random token
                random_token = random.choice(list(tokenizer.vocab.values()))
                # avoid special tokens for random selection
                while random_token < 5:  # indices 0-4 are special tokens in our vocab
                    random_token = random.choice(list(tokenizer.vocab.values()))
                input_ids[pos] = random_token
            else:
                # 10% keep original (no change to input_ids[pos])
                pass
        # Pad sequences to max_len
        pad_length = self.max_len - len(input_ids)
        if pad_length > 0:
            input_ids += [self.pad_id] * pad_length
            token_type_ids += [0] * pad_length
            attention_mask += [0] * pad_length
            mlm_labels += [-100] * pad_length

        # Convert to tensors
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        token_type_ids = torch.tensor(token_type_ids, dtype=torch.long)
        attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        mlm_labels = torch.tensor(mlm_labels, dtype=torch.long)
        nsp_label = torch.tensor(is_next, dtype=torch.long)
        return {"input_ids": input_ids, 
                "token_type_ids": token_type_ids, 
                "attention_mask": attention_mask, 
                "mlm_labels": mlm_labels, 
                "nsp_label": nsp_label}

# Instantiate the dataset and dataloader
pretrain_dataset = PretrainingDataset(khmer_sentences, english_sentences, tokenizer, max_seq_length=64)
train_loader = DataLoader(pretrain_dataset, batch_size=8, shuffle=True)

# Inspect a single sample from the dataset
sample = pretrain_dataset[0]
print("Sample input_ids (first 20):", sample["input_ids"][:20])
print("Sample token_type_ids (first 20):", sample["token_type_ids"][:20])
print("Sample attention_mask (first 20):", sample["attention_mask"][:20])
print("Sample mlm_labels (first 20):", sample["mlm_labels"][:20])
print("NSP label:", sample["nsp_label"].item())


Sample input_ids (first 20): tensor([ 2, 70, 47,  4, 68, 85,  3, 71, 93, 84, 78,  4, 60, 65, 82,  3,  0,  0,
         0,  0])
Sample token_type_ids (first 20): tensor([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
Sample attention_mask (first 20): tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
Sample mlm_labels (first 20): tensor([-100, -100, -100,   78, -100, -100, -100, -100, -100, -100, -100,   83,
        -100, -100, -100, -100, -100, -100, -100, -100])
NSP label: 0


## Defining the BERT Model (Transformer Encoder)

In [11]:
import torch.nn as nn
import torch.nn.functional as F

class TransformerEncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size, dropout_prob):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        # Multi-head self-attention weight matrices
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key   = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.out   = nn.Linear(hidden_size, hidden_size)
        # Feed-forward network layers
        self.ff_in = nn.Linear(hidden_size, intermediate_size)
        self.ff_out = nn.Linear(intermediate_size, hidden_size)
        # Layer norms and dropout
        self.norm1 = nn.LayerNorm(hidden_size, eps=1e-12)
        self.norm2 = nn.LayerNorm(hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(dropout_prob)
        self.dropout_attn = nn.Dropout(dropout_prob)

    def forward(self, x, attention_mask):
        # x shape: (batch, seq_len, hidden_size)
        batch_size, seq_len, _ = x.shape
        # Self-Attention
        # Project to queries, keys, values
        q = self.query(x)    # (batch, seq_len, hidden)
        k = self.key(x)      # (batch, seq_len, hidden)
        v = self.value(x)    # (batch, seq_len, hidden)
        # Reshape for multi-head: (batch, seq_len, num_heads, head_dim)
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)  # (batch, heads, seq_len, head_dim)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)  # (batch, heads, seq_len, head_dim)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)  # (batch, heads, seq_len, head_dim)
        # Compute scaled dot-product attention for each head
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (batch, heads, seq_len, seq_len)
        # Apply attention mask: mask is 0 for padded positions, 1 for real tokens
        # We want to mask out (set to -inf) positions that are pads (mask==0)
        if attention_mask is not None:
            # Expand mask to [batch, 1, 1, seq_len] for broadcasting
            attn_mask = attention_mask.view(batch_size, 1, 1, seq_len)
            attn_scores = attn_scores.masked_fill(attn_mask == 0, -1e9)
        attn_probs = F.softmax(attn_scores, dim=-1)  # (batch, heads, seq_len, seq_len)
        attn_probs = self.dropout_attn(attn_probs)   # dropout on attention weights
        attn_output = torch.matmul(attn_probs, v)    # (batch, heads, seq_len, head_dim)
        # Concatenate heads back together
        attn_output = attn_output.transpose(1, 2).reshape(batch_size, seq_len, self.hidden_size)
        # Apply output linear
        attn_output = self.out(attn_output)  # (batch, seq_len, hidden_size)
        attn_output = self.dropout(attn_output)
        # Residual connection + LayerNorm for attention sublayer
        x = self.norm1(x + attn_output)
        # Feed-Forward Network
        ff_output = self.ff_in(x)
        ff_output = F.gelu(ff_output)          # GELU activation
        ff_output = self.ff_out(ff_output)
        ff_output = self.dropout(ff_output)
        # Residual connection + LayerNorm for feed-forward sublayer
        x = self.norm2(x + ff_output)
        return x

class BertModel(nn.Module):
    def __init__(self, vocab_size, hidden_size=768, num_layers=12, num_heads=12, intermediate_size=3072, max_position_embeddings=512, dropout_prob=0.1):
        super().__init__()
        # Embeddings
        self.token_embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.position_embeddings = nn.Embedding(max_position_embeddings, hidden_size)
        self.segment_embeddings = nn.Embedding(2, hidden_size)  # segment ids 0 or 1
        self.embedding_layer_norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.embedding_dropout = nn.Dropout(dropout_prob)
        # Transformer encoder layers
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(hidden_size, num_heads, intermediate_size, dropout_prob) 
            for _ in range(num_layers)
        ])
        # Pooler: linear layer on CLS token representation
        self.pooler = nn.Linear(hidden_size, hidden_size)
        self.pooler_activation = nn.Tanh()

    def forward(self, input_ids, token_type_ids=None, attention_mask=None):
        # Get embeddings for each token id
        seq_length = input_ids.size(1)
        device = input_ids.device
        # Create position ids [0, 1, 2, ..., seq_length-1]
        position_ids = torch.arange(0, seq_length, dtype=torch.long, device=device).unsqueeze(0)  # (1, seq_len)
        position_ids = position_ids.expand(input_ids.size(0), seq_length)  # (batch, seq_len)
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)  # assume all 0 if not provided
        # Embedding lookup
        token_embeds = self.token_embeddings(input_ids)
        pos_embeds = self.position_embeddings(position_ids)
        seg_embeds = self.segment_embeddings(token_type_ids)
        # Sum token + position + segment embeddings
        x = token_embeds + pos_embeds + seg_embeds
        x = self.embedding_layer_norm(x)
        x = self.embedding_dropout(x)
        # Apply Transformer layers
        for layer in self.layers:
            x = layer(x, attention_mask)
        # x is now the final hidden states for each token (batch, seq_len, hidden)
        # Pooler: take hidden state of [CLS] (first token) and apply linear+tanh
        cls_token_hidden = x[:, 0]  # (batch, hidden_size)
        pooled_output = self.pooler_activation(self.pooler(cls_token_hidden))
        return x, pooled_output

# Instantiate the BERT model
vocab_size = len(tokenizer.vocab)
bert_model = BertModel(vocab_size=vocab_size, hidden_size=768, num_layers=12, num_heads=12, intermediate_size=3072, max_position_embeddings=64, dropout_prob=0.1)
# Test forward pass with a small batch
batch = next(iter(train_loader))
with torch.no_grad():
    seq_output, pooled_output = bert_model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
print("Batch sequence output shape:", seq_output.shape)   # (batch_size, seq_len, hidden_size)
print("Batch pooled output shape:", pooled_output.shape)  # (batch_size, hidden_size)


Batch sequence output shape: torch.Size([8, 64, 768])
Batch pooled output shape: torch.Size([8, 768])


## LightningModule for BERT Pretraining (MLM + NSP)

In [12]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.callbacks import ModelCheckpoint

class BertPretrainingLightningModule(pl.LightningModule):
    def __init__(self, vocab_size, hidden_size=768, num_layers=12, num_heads=12, intermediate_size=3072, max_seq_length=64, lr=1e-4, use_mlm=True, use_nsp=True):
        super().__init__()
        self.save_hyperparameters()  # saves hyperparams for checkpointing
        # Initialize the BertModel
        self.bert = BertModel(vocab_size, hidden_size, num_layers, num_heads, intermediate_size, max_position_embeddings=max_seq_length)
        # MLM head: linear layer tied with token embeddings weight
        self.mlm_head = nn.Linear(hidden_size, vocab_size)
        self.mlm_head.bias = nn.Parameter(torch.zeros(vocab_size))  # bias for MLM
        # Tie weights with embedding layer
        self.mlm_head.weight = self.bert.token_embeddings.weight
        # NSP head: linear layer for Next Sentence Prediction (2 classes)
        self.nsp_head = nn.Linear(hidden_size, 2)
        self.use_mlm = use_mlm
        self.use_nsp = use_nsp
        self.lr = lr

    def forward(self, input_ids, token_type_ids, attention_mask):
        seq_output, pooled_output = self.bert(input_ids, token_type_ids, attention_mask)
        # Compute MLM logits for each token in sequence
        mlm_logits = self.mlm_head(seq_output)  # shape (batch, seq_len, vocab_size)
        # Compute NSP logits using pooled output
        nsp_logits = self.nsp_head(pooled_output)  # shape (batch, 2)
        return mlm_logits, nsp_logits

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        token_type_ids = batch["token_type_ids"]
        attention_mask = batch["attention_mask"]
        mlm_labels = batch["mlm_labels"]
        nsp_label = batch["nsp_label"]
        mlm_logits, nsp_logits = self(input_ids, token_type_ids, attention_mask)
        # Calculate MLM loss (ignore_index=-100 to skip non-masked positions)
        loss = 0.0
        if self.use_mlm:
            mlm_loss = F.cross_entropy(mlm_logits.view(-1, mlm_logits.size(-1)), mlm_labels.view(-1), ignore_index=-100)
            loss += mlm_loss
            self.log("mlm_loss", mlm_loss, prog_bar=True, on_step=True, on_epoch=True)
        if self.use_nsp:
            nsp_loss = F.cross_entropy(nsp_logits, nsp_label)
            loss += nsp_loss
            self.log("nsp_loss", nsp_loss, prog_bar=True, on_step=True, on_epoch=True)
        # Log total loss
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        # Using Adam optimizer (could use AdamW for weight decay if desired)
        return torch.optim.Adam(self.parameters(), lr=self.lr)

# Initialize the LightningModule for pretraining
bert_pretrain_module = BertPretrainingLightningModule(vocab_size=len(tokenizer.vocab), lr=1e-4, use_mlm=True, use_nsp=True)


## Training the BERT Model (Pretraining)

In [13]:
# Set up MLflow logger and model checkpoint callback
mlflow_logger = MLFlowLogger(
    experiment_name="BERT-Khmer-Eng", 
    run_name="Pretraining",
    tracking_uri="https://mlflow.konai.dev/", #  "file:./logs/mlruns" | "https://mlflow.konai.dev/"
    tags={"model": "BERT"},    
)

checkpoint_callback = ModelCheckpoint(
    dirpath="pretrain_checkpoints", 
    filename="bert-pretrain-{epoch}", 
    save_top_k=3, 
    monitor="train_loss", 
    mode="min"
)

# Trainer configuration
trainer = pl.Trainer(
    max_epochs=100, 
    accelerator="gpu", 
    devices=[1],
    logger=mlflow_logger, 
    callbacks=[checkpoint_callback], 
    log_every_n_steps=1, 
    enable_progress_bar=True
)
# Note: For multiple GPUs, use Trainer(accelerator="gpu", devices=2, strategy="ddp") if available.

# Train the model
trainer.fit(bert_pretrain_module, train_loader)
print("Pretraining completed. Best checkpoint saved at:", checkpoint_callback.best_model_path)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Experiment with name BERT-Khmer-Eng not found. Creating it.
/home/acleda/miniconda3/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/acleda/DATA/code/projects/khmerbert/khmerbert/notebooks/exploratory/pretrain_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name     | Type      | Params | Mode 
-----------------------------------------------
0 | bert     | BertModel | 85.8 M | train
1 | mlm_head | Linear    | 73.8 K | train
2 | nsp_head | Linear    | 1.5 K  | train
-----------------------------------------------
85.8 M    Trainable params
0         Non-trainable params
85.8 M    Total params
343.091   Total estimated model params size (MB)
143       Modules in train mode
0         Modules in eval mode
/home/acleda/miniconda3/lib/python3.11/site-packages/pytorch_lightning/tra

Epoch 99: 100%|██████████| 5/5 [00:01<00:00,  3.13it/s, v_num=3612, mlm_loss_step=19.70, nsp_loss_step=0.761, train_loss_step=20.50, mlm_loss_epoch=22.30, nsp_loss_epoch=0.733, train_loss_epoch=23.00]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 5/5 [00:01<00:00,  2.58it/s, v_num=3612, mlm_loss_step=19.70, nsp_loss_step=0.761, train_loss_step=20.50, mlm_loss_epoch=22.30, nsp_loss_epoch=0.733, train_loss_epoch=23.00]
🏃 View run Pretraining at: https://mlflow.konai.dev/#/experiments/22/runs/7467fece04614cbdba0ecac979bf3612
🧪 View experiment at: https://mlflow.konai.dev/#/experiments/22
Pretraining completed. Best checkpoint saved at: /media/acleda/DATA/code/projects/khmerbert/khmerbert/notebooks/exploratory/pretrain_checkpoints/bert-pretrain-epoch=70.ckpt


## Fine-tuning Setup and Module

In [22]:
# ---------- universal helper -------------------------------------------------
def load_pretrained_bert(ckpt_path, tokenizer, max_pos=64):
    """
    Returns a BertModel initialised from the checkpoint's encoder weights.
    """
    ckpt = torch.load(ckpt_path, map_location="cpu")

    # Build a fresh BertModel with the SAME positional-embedding length
    bert = BertModel(
        vocab_size           = len(tokenizer.vocab),
        hidden_size          = ckpt["hyper_parameters"].get("hidden_size", 768),
        num_layers           = ckpt["hyper_parameters"].get("num_layers", 12),
        num_heads            = ckpt["hyper_parameters"].get("num_heads", 12),
        intermediate_size    = ckpt["hyper_parameters"].get("intermediate_size", 3072),
        max_position_embeddings = max_pos         # 64 in our pre-training
    )

    # Pull only the encoder weights out of the Lightning checkpoint
    enc_state = {k.replace("bert.", ""): v
                 for k, v in ckpt["state_dict"].items()
                 if k.startswith("bert.")}

    bert.load_state_dict(enc_state, strict=True)
    return bert
# -----------------------------------------------------------------------------


class TextClassificationLightningModule(pl.LightningModule):
    def __init__(self, num_classes, pretrained_model_path=None, lr=2e-5):
        super().__init__()
        self.save_hyperparameters()

        # ----- BERT encoder ---------------------------------------------------
        if pretrained_model_path:
            self.bert = load_pretrained_bert(
                pretrained_model_path, tokenizer, max_pos=64
            )
        else:
            self.bert = BertModel(
                vocab_size=len(tokenizer.vocab),
                max_position_embeddings=64
            )
        # ----------------------------------------------------------------------

        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, token_type_ids, attention_mask):
        _, pooled = self.bert(input_ids, token_type_ids, attention_mask)
        return self.classifier(self.dropout(pooled))

    def training_step(self, batch, _):
        ids, tt, mask, labels = batch
        loss = F.cross_entropy(self(ids, tt, mask), labels)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)



class SentencePairClassificationLightningModule(pl.LightningModule):
    def __init__(self, num_classes, pretrained_model_path=None, lr=2e-5):
        super().__init__()
        self.save_hyperparameters()

        if pretrained_model_path:
            self.bert = load_pretrained_bert(
                pretrained_model_path, tokenizer, max_pos=64
            )
        else:
            self.bert = BertModel(
                vocab_size=len(tokenizer.vocab),
                max_position_embeddings=64
            )

        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, token_type_ids, attention_mask):
        _, pooled = self.bert(input_ids, token_type_ids, attention_mask)
        return self.classifier(self.dropout(pooled))

    def training_step(self, batch, _):
        ids, tt, mask, labels = batch
        loss = F.cross_entropy(self(ids, tt, mask), labels)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)



class QuestionAnsweringLightningModule(pl.LightningModule):
    def __init__(self, pretrained_model_path=None, lr=3e-5):
        super().__init__()
        self.save_hyperparameters()

        if pretrained_model_path:
            self.bert = load_pretrained_bert(
                pretrained_model_path, tokenizer, max_pos=64
            )
        else:
            self.bert = BertModel(
                vocab_size=len(tokenizer.vocab),
                max_position_embeddings=64
            )

        # QA head: predicts start & end logits
        self.qa_outputs = nn.Linear(768, 2)

    def forward(self, input_ids, token_type_ids, attention_mask):
        seq_out, _ = self.bert(input_ids, token_type_ids, attention_mask)
        logits = self.qa_outputs(seq_out)           # (batch, seq_len, 2)
        return logits[..., 0], logits[..., 1]       # start, end

    def training_step(self, batch, _):
        ids, tt, mask, start_pos, end_pos = batch
        start_logits, end_logits = self(ids, tt, mask)
        loss = (F.cross_entropy(start_logits, start_pos) +
                F.cross_entropy(end_logits,   end_pos)) / 2
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


## Fine-tuning on Text Classifition

In [26]:
# Prepare data for text classification (language identification)
texts = khmer_sentences + english_sentences
labels = [0]*len(khmer_sentences) + [1]*len(english_sentences)  # 0 = Khmer, 1 = English
# Encode all texts
input_ids_list, token_type_ids_list, attention_mask_list = [], [], []
for txt in texts:
    ids, type_ids = tokenizer.encode(txt, add_special_tokens=True)
    # Pad to a fixed length (let's use 32 for classification tasks for speed)
    if len(ids) < 32:
        pad_len = 32 - len(ids)
        attention_mask = [1]*len(ids) + [0]*pad_len
        type_ids = type_ids + [0]*pad_len
        ids = ids + [tokenizer.vocab["[PAD]"]]*pad_len
    else:
        ids = ids[:32]
        type_ids = type_ids[:32]
        attention_mask = [1]*32
    input_ids_list.append(ids)
    token_type_ids_list.append(type_ids)
    attention_mask_list.append(attention_mask)
# Convert to tensors
input_ids_tensor = torch.tensor(input_ids_list, dtype=torch.long)
token_type_ids_tensor = torch.tensor(token_type_ids_list, dtype=torch.long)
attention_mask_tensor = torch.tensor(attention_mask_list, dtype=torch.long)
labels_tensor = torch.tensor(labels, dtype=torch.long)
# Create DataLoader
clf_dataset = torch.utils.data.TensorDataset(input_ids_tensor, token_type_ids_tensor, attention_mask_tensor, labels_tensor)
clf_loader = DataLoader(clf_dataset, batch_size=4, shuffle=True)

# Initialize the text classification module with pretrained weights
text_clf_module = TextClassificationLightningModule(
    num_classes=2, 
    pretrained_model_path=checkpoint_callback.best_model_path, 
    lr=2e-5
)

checkpoint_callback_clf = ModelCheckpoint(
    dirpath="clf_checkpoints", 
    filename="best-textclf", 
    save_top_k=1, 
    monitor="train_loss", 
    mode="min"
)

trainer_clf = pl.Trainer(
    max_epochs=3, 
    logger=MLFlowLogger(
        experiment_name="BERT-Khmer-Eng", 
        run_name="FineTune-TextClass",
        tracking_uri="https://mlflow.konai.dev/",
        ), 
    callbacks=[checkpoint_callback_clf], 
    log_every_n_steps=1, 
    enable_progress_bar=False
)

trainer_clf.fit(text_clf_module, clf_loader)
print("Text classification fine-tuning done. Best model:", checkpoint_callback_clf.best_model_path)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/acleda/miniconda3/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/acleda/DATA/code/projects/khmerbert/khmerbert/notebooks/exploratory/clf_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name       | Type      | Params | Mode 
-------------------------------------------------
0 | bert       | BertModel | 85.8 M | train
1 | dropout    | Dropout   | 0      | train
2 | classifier | Linear    | 1.5 K  | train
-------------------------------------------------
85.8 M    Trainable params
0         Non

🏃 View run FineTune-TextClass at: https://mlflow.konai.dev/#/experiments/22/runs/eb5b7237ab4b4e50bdd7e1b8b46c071f
🧪 View experiment at: https://mlflow.konai.dev/#/experiments/22
Text classification fine-tuning done. Best model: /media/acleda/DATA/code/projects/khmerbert/khmerbert/notebooks/exploratory/clf_checkpoints/best-textclf-v1.ckpt


## Fine-tuning on Sentence Pair Classification

In [27]:
import itertools
random.seed(42)
# Generate sentence pair examples
same_lang_pairs = []
diff_lang_pairs = []
# All possible pairs within Khmer and within English
khmer_pairs = list(itertools.combinations(khmer_sentences, 2))[:5]   # take first 5 pairs
english_pairs = list(itertools.combinations(english_sentences, 2))[:5]
for a, b in khmer_pairs + english_pairs:
    same_lang_pairs.append((a, b, 1))
# Mixed language pairs (Khmer, English)
for i in range(5):
    a = random.choice(khmer_sentences)
    b = random.choice(english_sentences)
    diff_lang_pairs.append((a, b, 0))
pairs_data = same_lang_pairs + diff_lang_pairs
# Encode pairs
input_ids_list, token_type_ids_list, attention_mask_list, labels_list = [], [], [], []
for sent_a, sent_b, label in pairs_data:
    ids, type_ids = tokenizer.encode(sent_a, sent_b, add_special_tokens=True)
    # Pad to length 32
    if len(ids) < 32:
        pad_len = 32 - len(ids)
        attention_mask = [1]*len(ids) + [0]*pad_len
        type_ids = type_ids + [0]*pad_len  # pad segment ids as 0 (doesn't matter)
        ids = ids + [tokenizer.vocab["[PAD]"]]*pad_len
    else:
        ids = ids[:32]
        type_ids = type_ids[:32]
        attention_mask = [1]*32
    input_ids_list.append(ids)
    token_type_ids_list.append(type_ids)
    attention_mask_list.append(attention_mask)
    labels_list.append(label)
# Convert to tensors
input_ids_tensor = torch.tensor(input_ids_list, dtype=torch.long)
token_type_ids_tensor = torch.tensor(token_type_ids_list, dtype=torch.long)
attention_mask_tensor = torch.tensor(attention_mask_list, dtype=torch.long)
labels_tensor = torch.tensor(labels_list, dtype=torch.long)
pair_dataset = torch.utils.data.TensorDataset(input_ids_tensor, token_type_ids_tensor, attention_mask_tensor, labels_tensor)
pair_loader = DataLoader(pair_dataset, batch_size=4, shuffle=True)

# Initialize the sentence pair classification module and train
pair_clf_module = SentencePairClassificationLightningModule(
    num_classes=2, 
    pretrained_model_path=checkpoint_callback.best_model_path, 
    lr=2e-5
)

checkpoint_callback_pair = ModelCheckpoint(
    dirpath="pair_checkpoints", 
    filename="best-sentencepair", 
    save_top_k=1, monitor="train_loss", 
    mode="min"
)

trainer_pair = pl.Trainer(
    max_epochs=3, 
    logger=MLFlowLogger(
        experiment_name="BERT-Khmer-Eng", 
        run_name="FineTune-SentencePair", 
        tracking_uri="https://mlflow.konai.dev/",
    ), 
    callbacks=[checkpoint_callback_pair], 
    log_every_n_steps=1, 
    enable_progress_bar=False
)

trainer_pair.fit(pair_clf_module, pair_loader)

print("Sentence pair classification fine-tuning done. Best model:", checkpoint_callback_pair.best_model_path)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/acleda/miniconda3/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/acleda/DATA/code/projects/khmerbert/khmerbert/notebooks/exploratory/pair_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name       | Type      | Params | Mode 
-------------------------------------------------
0 | bert       | BertModel | 85.8 M | train
1 | dropout    | Dropout   | 0      | train
2 | classifier | Linear    | 1.5 K  | train
-------------------------------------------------
85.8 M    Trainable params
0         No

🏃 View run FineTune-SentencePair at: https://mlflow.konai.dev/#/experiments/22/runs/8e04d9106706444086c3c6874a24f774
🧪 View experiment at: https://mlflow.konai.dev/#/experiments/22
Sentence pair classification fine-tuning done. Best model: /media/acleda/DATA/code/projects/khmerbert/khmerbert/notebooks/exploratory/pair_checkpoints/best-sentencepair-v1.ckpt


## Fine-tuning on Question Answering (Span Prediction)

In [28]:
# Prepare QA examples
qa_examples = []
# Khmer example
context_km = "ក្រុង​ភ្នំពេញ​គឺ​ជា​រណ្តៅ​សេដ្ឋកិច្ច​របស់​ប្រទេស​កម្ពុជា។"
question_km = "ក្រុង​ណា​ជា​រណ្តៅ​សេដ្ឋកិច្ច​របស់​ប្រទេស​កម្ពុជា?"
answer_km = "ក្រុង​ភ្នំពេញ"
qa_examples.append((question_km, context_km, answer_km))
# English example
context_en = "Alice went to the market in Phnom Penh."
question_en = "Where did Alice go?"
answer_en = "to the market in Phnom Penh"
qa_examples.append((question_en, context_en, answer_en))

input_ids_list, token_type_ids_list, attention_mask_list = [], [], []
start_positions, end_positions = [], []
for question, context, answer in qa_examples:
    # Encode question-context pair
    ids, type_ids = tokenizer.encode(question, context, add_special_tokens=True)
    # Determine position of answer in tokenized context
    # Find the index of the first [SEP] to know when context starts
    sep_index = ids.index(tokenizer.vocab["[SEP]"])
    # Context tokens start after first [SEP]
    context_start = sep_index + 1
    context_tokens = tokenizer.tokenize(context)
    answer_tokens = tokenizer.tokenize(answer)
    # Find sublist answer_tokens in context_tokens
    # We'll do a simple search
    answer_len = len(answer_tokens)
    start_idx_in_context = -1
    for i in range(len(context_tokens) - answer_len + 1):
        if context_tokens[i:i+answer_len] == answer_tokens:
            start_idx_in_context = i
            break
    if start_idx_in_context == -1:
        start_idx = 0
        end_idx = 0
    else:
        start_idx = context_start + start_idx_in_context
        end_idx = start_idx + answer_len - 1
    # Pad to 64
    if len(ids) < 64:
        pad_len = 64 - len(ids)
        attention_mask = [1]*len(ids) + [0]*pad_len
        type_ids = type_ids + [1]*pad_len  # pad segment as context (segment 1)
        ids = ids + [tokenizer.vocab["[PAD]"]]*pad_len
    else:
        ids = ids[:64]
        type_ids = type_ids[:64]
        attention_mask = [1]*64
        # Ensure [SEP] at end
        ids[-1] = tokenizer.vocab["[SEP]"]
        type_ids[-1] = 1
        if end_idx >= 64:
            end_idx = 63
        if start_idx >= 64:
            start_idx = 63
    input_ids_list.append(ids)
    token_type_ids_list.append(type_ids)
    attention_mask_list.append(attention_mask)
    start_positions.append(start_idx)
    end_positions.append(end_idx)

input_ids_tensor = torch.tensor(input_ids_list, dtype=torch.long)
token_type_ids_tensor = torch.tensor(token_type_ids_list, dtype=torch.long)
attention_mask_tensor = torch.tensor(attention_mask_list, dtype=torch.long)
start_tensor = torch.tensor(start_positions, dtype=torch.long)
end_tensor = torch.tensor(end_positions, dtype=torch.long)
qa_dataset = torch.utils.data.TensorDataset(input_ids_tensor, token_type_ids_tensor, attention_mask_tensor, start_tensor, end_tensor)
qa_loader = DataLoader(qa_dataset, batch_size=2, shuffle=True)

# Fine-tune QA module
qa_module = QuestionAnsweringLightningModule(
    pretrained_model_path=checkpoint_callback.best_model_path, 
    lr=3e-5
)

checkpoint_callback_qa = ModelCheckpoint(
    dirpath="qa_checkpoints", 
    filename="best-qa", 
    save_top_k=1, 
    monitor="train_loss", 
    mode="min"
)

trainer_qa = pl.Trainer(
    max_epochs=5, 
    logger=MLFlowLogger(
        experiment_name="BERT-Khmer-Eng", 
        run_name="FineTune-QA",
        tracking_uri="https://mlflow.konai.dev/",
    ), 
    callbacks=[checkpoint_callback_qa], 
    log_every_n_steps=1, 
    enable_progress_bar=False
)

trainer_qa.fit(qa_module, qa_loader)
print("QA fine-tuning done. Best model:", checkpoint_callback_qa.best_model_path)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/acleda/miniconda3/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/acleda/DATA/code/projects/khmerbert/khmerbert/notebooks/exploratory/qa_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name       | Type      | Params | Mode 
-------------------------------------------------
0 | bert       | BertModel | 85.8 M | train
1 | qa_outputs | Linear    | 1.5 K  | train
-------------------------------------------------
85.8 M    Trainable params
0         Non-trainable params
85.8 M    Total params
343.

🏃 View run FineTune-QA at: https://mlflow.konai.dev/#/experiments/22/runs/cd6ce68e94a44d658232f4653a05c549
🧪 View experiment at: https://mlflow.konai.dev/#/experiments/22
QA fine-tuning done. Best model: /media/acleda/DATA/code/projects/khmerbert/khmerbert/notebooks/exploratory/qa_checkpoints/best-qa-v1.ckpt
